In [2]:
%load_ext autoreload
%autoreload 2

import sentiments_utils as utils
import torch
import pandas as pd
from transformers import pipeline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# Dataset configuration
DATASET_PATH = "projected_datasets"
DATASET_SPLITS = {"train": f"{DATASET_PATH}/train-robertuito.parquet",
                  "test_validation": f"{DATASET_PATH}/validation-reviewed.parquet"}
# Language configuration
SOURCE_COLUMN = "shp" 
TARGET_COLUMN = "spa"
LABEL_COLUMN = "label"
NUMBER_LABELS = 3  # Positive, Negative, Neutral
# General configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
# Cargar datasets
full_dataset = utils.load_full_dataset(DATASET_PATH, DATASET_SPLITS)
train_dataset = full_dataset["train"]
test_validation_dataset = full_dataset["test_validation"]

# Limpiar datasets
train_dataset = utils.dedupe_bilingual(train_dataset, langs=(SOURCE_COLUMN, TARGET_COLUMN))
test_validation_dataset = utils.dedupe_bilingual(test_validation_dataset, langs=(SOURCE_COLUMN, TARGET_COLUMN))
# Remover overlaps entre train y validation
train_dataset = utils.remove_overlaps_bilingual(train_dataset, test_validation_dataset, langs=(SOURCE_COLUMN, TARGET_COLUMN))

# Separar test y validation usando 50% para cada uno
test_validation_dataset = test_validation_dataset.train_test_split(test_size=0.5, seed=42)
test_dataset = test_validation_dataset["test"]
validation_dataset = test_validation_dataset["train"]

train_dataset, test_dataset, validation_dataset


📥 Loading dataset: projected_datasets

📊 Dataset Statistics:
   train: 16505 examples
   Example Shipibo: Jato shinamawe mesko yokabo axon neskaakin....
   Example Spanish: Ahora hazles recordar a través de diferentes pregu...
   test_validation: 1924 examples
   Example Shipibo: Metsara iwanke....
   Example Spanish: Fue maravilloso....


Filter:   0%|          | 0/10080 [00:00<?, ? examples/s]

(Dataset({
     features: ['spa', 'shp', '__index_level_0__', 'label', 'sentiment_score'],
     num_rows: 8613
 }),
 Dataset({
     features: ['spa', 'shp', '__index_level_0__', 'label', 'sentiment_score'],
     num_rows: 921
 }),
 Dataset({
     features: ['spa', 'shp', '__index_level_0__', 'label', 'sentiment_score'],
     num_rows: 920
 }))

# Modelo 1: XLM-Roberta-Base Shipibo

In [5]:
# Modelo
model_1_name = "xlm-roberta-sentiment-shp"
model_1_path = "models/xlm-roberta-sentiment-shp"

# Tokenizer
tokenizer_1_name = "xlm-roberta-sentiment-shp"
tokenizer_1_path = "tokenizers/xlm-roberta-sentiment-shp"

In [6]:
model_1 = utils.prepare_model(model_path=model_1_path, num_labels=NUMBER_LABELS)
tokenizer_1 = utils.prepare_tokenizer(tokenizer_path=tokenizer_1_path)
pipeline_1 = pipeline(task="text-classification", model=model_1, tokenizer=tokenizer_1, device=DEVICE)


⬇️  Loading model: models/xlm-roberta-sentiment-shp
   Model loaded from: models/xlm-roberta-sentiment-shp

🔧 Loading tokenizer: tokenizers/xlm-roberta-sentiment-shp


Device set to use cuda


   Tokenizer loaded from: tokenizers/xlm-roberta-sentiment-shp


In [7]:
# Evaluar modelo con dataset de validacion
results_1 = utils.evaluate_dataset(pipeline_1, test_dataset, source_column=SOURCE_COLUMN, label_column=LABEL_COLUMN)
results_1

Evaluating: 100%|██████████| 58/58 [00:07<00:00,  7.80it/s]


{'accuracy': 0.7752442996742671,
 'balanced_accuracy': 0.7168652239147367,
 'precision': 0.780187292494256,
 'recall': 0.7752442996742671,
 'f1': 0.775707664753159,
 'confusion_matrix': array([[196,  44,   4],
        [ 68, 456,  37],
        [ 24,  30,  62]])}

### Modelo 2: MMBert

In [8]:
# Modelo
model_2_name = "mmBERT-sentiment-shp"
model_2_path = "models/mmBERT-sentiment-shp"

# Tokenizer
tokenizer_2_name = "mmBERT-sentiment-shp"
tokenizer_2_path = "tokenizers/mmBERT-sentiment-shp"

In [9]:
model_2 = utils.prepare_model(model_path=model_2_path, num_labels=NUMBER_LABELS)
tokenizer_2 = utils.prepare_tokenizer(tokenizer_path=tokenizer_2_path)
pipeline_2 = pipeline(task="text-classification", model=model_2, tokenizer=tokenizer_2, device=DEVICE)


⬇️  Loading model: models/mmBERT-sentiment-shp
   Model loaded from: models/mmBERT-sentiment-shp

🔧 Loading tokenizer: tokenizers/mmBERT-sentiment-shp


Device set to use cuda


   Tokenizer loaded from: tokenizers/mmBERT-sentiment-shp


In [10]:
# Evaluar modelo con dataset de validacion
results_2 = utils.evaluate_dataset(pipeline_2, test_dataset, source_column=SOURCE_COLUMN, label_column=LABEL_COLUMN)
results_2

Evaluating:   0%|          | 0/58 [00:00<?, ?it/s]/home/chech/torch/lib/python3.12/site-packages/torch/_inductor/compile_fx.py:282: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
W1119 16:48:03.726000 90728 torch/_inductor/utils.py:1436] [1/0_1] Not enough SMs to use max_autotune_gemm mode
Evaluating: 100%|██████████| 58/58 [00:23<00:00,  2.42it/s]


{'accuracy': 0.7350705754614549,
 'balanced_accuracy': 0.6840734683650256,
 'precision': 0.7492701904266704,
 'recall': 0.7350705754614549,
 'f1': 0.7396726394961425,
 'confusion_matrix': array([[182,  44,  18],
        [ 81, 433,  47],
        [ 23,  31,  62]])}